![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 03 | Evaluate — the Report

*Project 3 — AI Product: From Model to Product*

---

**This notebook is your report.** It is read on its own, by someone who has not seen notebooks 01 and 02: the question, the evidence, what you concluded and how far to trust it.

> **Done when:** the final version is evaluated once, its failures are analysed with examples, its cost is known, and the conclusions answer the question you framed in notebook 01.

---
## 0. Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.functions import *

pd.set_option("display.max_colwidth", 200)
sns.set_theme(style="whitegrid")

---
## 1. The question

The user, the decision, the metric and the baseline score, in a few sentences, as framed in notebook 01.

---
## 2. The final evaluation

- **Model track:** the final model, evaluated **once** on the test set, compared with its validation score and with the baseline.
- **LLM track:** the whole evaluation set. Report separately: the retrieval hit rate (or the share of questions with the right tools), whether answers are correct and cite the right source, and whether the out-of-scope questions are refused.

In [3]:
import pandas as pd
import json

# Fixed evaluation jobs
jobs = pd.read_csv(
    "../data/evaluation_jobs.csv",
    dtype={"id": str}
)

# Your manually assigned relevance labels
labels = pd.read_csv(
    "../data/relevance_labels.csv",
    dtype={"job_id": str}
)

# Saved zero-shot predictions
zero_shot_scores = pd.read_csv(
    "../data/zero_shot_scores.csv",
    dtype={"job_id": str}
)

# Candidate profiles
with open("../data/candidate_profiles.json", encoding="utf-8") as f:
    candidate_profiles = json.load(f)

print("Jobs:", len(jobs))
print("Labels:", len(labels))
print("Zero-shot scores:", len(zero_shot_scores))
print("Candidate profiles:", len(candidate_profiles))

Jobs: 28
Labels: 140
Zero-shot scores: 140
Candidate profiles: 5


In [4]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Combine job titles and descriptions
jobs["job_text"] = (
    jobs["title"].fillna("") + " " +
    jobs["description"].fillna("")
)

# Same job dataset for every model
job_texts = jobs["job_text"].tolist()

# Store predictions
tfidf_results = []

start = time.perf_counter()

for profile_id, candidate_text in candidate_profiles.items():

    documents = [candidate_text] + job_texts

    vectorizer = TfidfVectorizer(stop_words="english")

    matrix = vectorizer.fit_transform(documents)

    scores = cosine_similarity(
        matrix[0:1],
        matrix[1:]
    ).flatten()

    for job_id, score in zip(jobs["id"], scores):
        tfidf_results.append({
            "profile_id": profile_id,
            "job_id": str(job_id),
            "tfidf_score": score
        })

tfidf_time = time.perf_counter() - start

tfidf_scores = pd.DataFrame(tfidf_results)

print("TF-IDF predictions:", len(tfidf_scores))
print(f"Execution time: {tfidf_time:.3f} seconds")

TF-IDF predictions: 140
Execution time: 0.029 seconds


In [5]:
import time

In [6]:
from sentence_transformers import SentenceTransformer

start = time.perf_counter()

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Encode job descriptions once
job_embeddings = model.encode(
    job_texts,
    show_progress_bar=True
)

# Encode all five candidate profiles
profile_ids = list(candidate_profiles.keys())

candidate_embeddings = model.encode(
    list(candidate_profiles.values()),
    show_progress_bar=True
)

# Compare each candidate against all jobs
similarity_matrix = cosine_similarity(
    candidate_embeddings,
    job_embeddings
)

semantic_results = []

for i, profile_id in enumerate(profile_ids):
    for j, job_id in enumerate(jobs["id"]):
        semantic_results.append({
            "profile_id": profile_id,
            "job_id": str(job_id),
            "semantic_score": similarity_matrix[i, j]
        })

semantic_time = time.perf_counter() - start

semantic_scores = pd.DataFrame(semantic_results)

print("Semantic predictions:", len(semantic_scores))
print(f"Execution time: {semantic_time:.3f} seconds")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Semantic predictions: 140
Execution time: 3.477 seconds


In [7]:
from sentence_transformers import SentenceTransformer

start = time.perf_counter()

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Encode job descriptions once
job_embeddings = model.encode(
    job_texts,
    show_progress_bar=True
)

# Encode all five candidate profiles
profile_ids = list(candidate_profiles.keys())

candidate_embeddings = model.encode(
    list(candidate_profiles.values()),
    show_progress_bar=True
)

# Compare each candidate against all jobs
similarity_matrix = cosine_similarity(
    candidate_embeddings,
    job_embeddings
)

semantic_results = []

for i, profile_id in enumerate(profile_ids):
    for j, job_id in enumerate(jobs["id"]):
        semantic_results.append({
            "profile_id": profile_id,
            "job_id": str(job_id),
            "semantic_score": similarity_matrix[i, j]
        })

semantic_time = time.perf_counter() - start

semantic_scores = pd.DataFrame(semantic_results)

print("Semantic predictions:", len(semantic_scores))
print(f"Execution time: {semantic_time:.3f} seconds")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Semantic predictions: 140
Execution time: 2.864 seconds


In [8]:
comparison_df = (
    labels[["profile_id", "job_id", "relevance"]]
    .merge(
        tfidf_scores,
        on=["profile_id", "job_id"],
        how="left"
    )
    .merge(
        semantic_scores,
        on=["profile_id", "job_id"],
        how="left"
    )
    .merge(
        zero_shot_scores,
        on=["profile_id", "job_id"],
        how="left"
    )
)

print("Comparison shape:", comparison_df.shape)

display(comparison_df.head())

Comparison shape: (140, 6)


,profile_id,job_id,relevance,tfidf_score,semantic_score,zero_shot_score
0,ml_engineer,5953648768,1,0.135777,0.431707,0.613409
1,ml_engineer,5943389711,1,0.017025,0.258443,0.672074
2,ml_engineer,5956672032,2,0.160443,0.613873,0.510513
3,ml_engineer,5939347586,1,0.043640,0.408838,0.621427
4,ml_engineer,5892995304,1,0.121569,0.357741,0.626463


In [9]:
import numpy as np
from sklearn.metrics import ndcg_score

def evaluate_model(df, score_column, k=5):
    results = []

    for profile_id, group in df.groupby("profile_id"):

        # Sort jobs by predicted score
        ranked = group.sort_values(
            score_column,
            ascending=False
        )

        # Precision@5: only relevance=2 counts
        precision = (
            (ranked["relevance"].head(k) == 2).sum() / k
        )

        # nDCG@5: uses all relevance levels (0, 1, 2)
        ndcg = ndcg_score(
            [group["relevance"].to_numpy()],
            [group[score_column].to_numpy()],
            k=k
        )

        results.append({
            "profile_id": profile_id,
            "model": score_column,
            "precision_at_5": precision,
            "ndcg_at_5": ndcg
        })

    return pd.DataFrame(results)

In [10]:
tfidf_eval = evaluate_model(
    comparison_df, "tfidf_score"
)

semantic_eval = evaluate_model(
    comparison_df, "semantic_score"
)

zero_shot_eval = evaluate_model(
    comparison_df, "zero_shot_score"
)

evaluation_results = pd.concat(
    [tfidf_eval, semantic_eval, zero_shot_eval],
    ignore_index=True
)

display(evaluation_results)

,profile_id,model,precision_at_5,ndcg_at_5
0,backend_developer,tfidf_score,0.2,0.862224
1,cloud_engineer,tfidf_score,0.4,0.763701
2,data_analyst,tfidf_score,0.2,0.808552
3,ml_engineer,tfidf_score,0.4,0.803126
4,technical_support,tfidf_score,0.0,0.137776
5,backend_developer,semantic_score,0.4,0.915523
6,cloud_engineer,semantic_score,0.6,0.915211
7,data_analyst,semantic_score,0.2,0.757836
8,ml_engineer,semantic_score,0.6,0.742945
9,technical_support,semantic_score,0.4,0.575417


In [11]:
label_distribution = pd.crosstab(
    comparison_df["profile_id"],
    comparison_df["relevance"]
)

display(label_distribution)

relevance,0,1,2
profile_id,,,
backend_developer,17,9,2
cloud_engineer,18,7,3
data_analyst,22,5,1
ml_engineer,17,8,3
technical_support,22,4,2


In [12]:
comparison_df = (
    labels[["profile_id", "job_id", "relevance"]]
    .merge(tfidf_scores, on=["profile_id", "job_id"])
    .merge(semantic_scores, on=["profile_id", "job_id"])
    .merge(zero_shot_scores, on=["profile_id", "job_id"])
)

print("Comparison shape:", comparison_df.shape)

Comparison shape: (140, 6)


In [13]:
tfidf_eval = evaluate_model(comparison_df, "tfidf_score")
semantic_eval = evaluate_model(comparison_df, "semantic_score")
zero_shot_eval = evaluate_model(comparison_df, "zero_shot_score")

evaluation_results = pd.concat(
    [tfidf_eval, semantic_eval, zero_shot_eval],
    ignore_index=True
)

summary = (
    evaluation_results
    .groupby("model")
    .agg(
        mean_precision_at_5=("precision_at_5", "mean"),
        mean_ndcg_at_5=("ndcg_at_5", "mean"),
        std_ndcg_at_5=("ndcg_at_5", "std")
    )
    .sort_values("mean_ndcg_at_5", ascending=False)
)

display(summary.round(3))

,mean_precision_at_5,mean_ndcg_at_5,std_ndcg_at_5
model,,,
semantic_score,0.44,0.781,0.142
tfidf_score,0.24,0.675,0.302
zero_shot_score,0.08,0.242,0.232


---
## 3. Where it goes wrong

- **Model track:** a confusion matrix, per-class precision and recall, and a sample of mistakes **read in full**, with your reading of why.
- **LLM track:** every failure classified as a retrieval (or tool) failure or a generation failure, with examples.

Then: where should your product **not** be trusted?

In [14]:
# Add job titles to comparison table
error_df = comparison_df.merge(
    jobs[["id", "title"]],
    left_on="job_id",
    right_on="id",
    how="left"
)

# Calculate rank for each candidate
error_df["semantic_rank"] = (
    error_df.groupby("profile_id")["semantic_score"]
    .rank(method="first", ascending=False)
)

# Find irrelevant jobs in the top 5
errors = error_df[
    (error_df["semantic_rank"] <= 5)
    & (error_df["relevance"] == 0)
]

display(
    errors[
        ["profile_id", "title", "semantic_rank", "semantic_score", "relevance"]
    ].sort_values(["profile_id", "semantic_rank"])
)

,profile_id,title,semantic_rank,semantic_score,relevance
53,backend_developer,Aftersales Technical Support Specialist (m/w/d),5.0,0.401243,0
93,cloud_engineer,.NET Software Engineer,4.0,0.576002,0
72,data_analyst,"Referent der Geschäftsführung (m/w/d): Datenanalyse für Unternehmenssteuerung & Reporting mit SQL, Azure & Power BI",5.0,0.507056,0
14,ml_engineer,Junior Data Analyst / Mitarbeiter Digitalisierung m/w/d,2.0,0.548690,0
17,ml_engineer,"Deine Chance: Datenmanagement Upskilling - SQL, Microsoft Azure & Power BI für den schnellen Wiedereinstieg",4.0,0.499525,0
139,technical_support,"Werde IT Support Specialist (m/w/d), Quereinsteiger willkommen!",2.0,0.564478,0
127,technical_support,Softwareentwickler / Developer (m/w/d) Business Intelligence / Data Warehouse,3.0,0.529866,0


In [15]:
from pathlib import Path

output_dir = Path("../data/results")
output_dir.mkdir(parents=True, exist_ok=True)

evaluation_results.to_csv(
    output_dir / "evaluation_results.csv",
    index=False
)

summary.to_csv(
    output_dir / "model_summary.csv"
)

print("Evaluation results saved!")

Evaluation results saved!


### Error Analysis

Sentence Transformers achieved the best overall ranking performance, with mean nDCG@5 of **0.781** and Precision@5 of **0.44**.

Across five candidate profiles, seven irrelevant vacancies appeared in the top-five recommendations.

The analysis revealed three major limitations:

1. **Occupation mismatch:** Semantically related technical skills do not always indicate compatible job responsibilities.
2. **Retrieval noise:** The dataset contains advertisements for training programs that are not necessarily genuine employment opportunities.
3. **Insufficient candidate–job constraints:** The model does not explicitly verify seniority, employment type, or essential qualifications.

These findings suggest that combining semantic ranking with vacancy validation and structured filtering could improve recommendation quality.

The evaluation was conducted on 28 fixed job listings and five fictional candidate profiles using AI-assisted relevance labels. The results should therefore be interpreted as exploratory rather than representative of real-world recruitment performance.

---
## 4. Cost and speed

- **Model track:** training time, model size on disk, time to predict one text.
- **LLM track:** tokens, cost and time per question, and the total for the evaluation. What would 1,000 questions a day cost per month?

---
## 5. The saved model or pipeline

Reload what you saved in notebook 02 and use it on a new example, exactly as the app does.

---
## 6. Conclusions and limitations

Would you ship it, to whom, and with what warning? What data, products or questions does it hold for? What would you do next?